# Бизнес эффект

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px

from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

from genre_ensemble import GenreStackingClassifier

sys.path.insert(0, str(Path.cwd()))

DATA_PATH = Path("../../spotify_data.csv")
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = Path("../models")
ENSEMBLE_DIR = MODELS_DIR / "ensemble"
for d in (DATA_DIR, SPLITS_DIR, MODELS_DIR, ENSEMBLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 123
FEATURES = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "time_signature",
]
EXTRA_COLS = ["popularity", "year", "track_id", "artist_name", "track_name"]

In [2]:
test_df = pl.read_parquet(SPLITS_DIR / "test.parquet")
classes = np.load(MODELS_DIR / "label_classes.npy", allow_pickle=True).tolist()
shap_global = pl.read_parquet(DATA_DIR / "shap_global.parquet")
ensemble = GenreStackingClassifier.load(ENSEMBLE_DIR)
print(f"test_rows={test_df.height}  classes={len(classes)}")

test_rows=173965  classes=82


In [3]:
test_pdf = test_df.to_pandas()
proba = ensemble.predict_proba(test_pdf, calibrated=True)
pred_idx = np.argmax(proba, axis=1)
pred_label = np.array(classes)[pred_idx]
confidence = proba.max(axis=1)

scored = test_df.with_columns([
    pl.Series("predicted_genre", pred_label),
    pl.Series("confidence", confidence),
    pl.Series("agrees_with_tag", pred_label == test_df["genre"].to_numpy()),
])
print(f"tag_agreement = {scored['agrees_with_tag'].mean()}")

tag_agreement = 0.2843100623688673


In [4]:
fig = px.bar(
    shap_global.to_pandas().sort_values("mean_abs_shap"),
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title="Global SHAP feature importance",
)
fig.update_layout(height=520)
fig.show()

In [5]:
top5 = shap_global.sort("mean_abs_shap", descending=True).head(5)["feature"].to_list()
print("top_5_features =", top5)

top_5_features = ['instrumentalness', 'acousticness', 'danceability', 'energy', 'duration_ms']


In [6]:
baseline = 0.50
mde = 0.02
effect = mde / np.sqrt(baseline * (1 - baseline))
n_per_arm = NormalIndPower().solve_power(
    effect_size=effect, alpha=0.05, power=0.8, alternative="two-sided"
)
print(f"required_n_per_arm = {n_per_arm}")

required_n_per_arm = 9811.075636657808


In [7]:
N_PLAYLISTS = 200
PLAYLIST_SIZE = 50
CONFIDENCE_TAU = 0.30
rng = np.random.default_rng(SEED)

eligible = (
    scored
    .group_by("genre")
    .len()
    .filter(pl.col("len") >= PLAYLIST_SIZE * 4)
    .sort("len", descending=True)
    .head(60)["genre"]
    .to_list()
)
ctrl_pool = scored.filter(pl.col("genre").is_in(eligible))
treat_pool = scored.filter(
    pl.col("predicted_genre").is_in(eligible) & (pl.col("confidence") >= CONFIDENCE_TAU)
)
print(
    f"eligible_genres={len(eligible)}  control_pool={ctrl_pool.height}  treatment_pool={treat_pool.height}"
)


def build_playlists(pool, key_col, n, size, seed):
    pdf = pool.to_pandas()
    rng = np.random.default_rng(seed)
    rows = []
    genres = pdf[key_col].unique().tolist()
    for i in range(n):
        g = rng.choice(genres)
        cands = pdf[pdf[key_col] == g]
        if len(cands) < size:
            continue
        picked = cands.sample(n=size, random_state=int(rng.integers(0, 2**31 - 1)))
        rows.append({
            "playlist_id": i,
            "genre": g,
            "mean_popularity": picked["popularity"].mean(),
            "median_popularity": picked["popularity"].median(),
        })
    return pd.DataFrame(rows)


ctrl_pl = build_playlists(ctrl_pool, "genre", N_PLAYLISTS, PLAYLIST_SIZE, SEED)
treat_pl = build_playlists(
    treat_pool, "predicted_genre", N_PLAYLISTS, PLAYLIST_SIZE, SEED + 1
)
print(f"control_playlists={len(ctrl_pl)}  treatment_playlists={len(treat_pl)}")

eligible_genres=60  control_pool=150538  treatment_pool=47707
control_playlists=200  treatment_playlists=180


In [8]:
ctrl_pop = ctrl_pl["mean_popularity"].to_numpy()
treat_pop = treat_pl["mean_popularity"].to_numpy()
t_stat, t_p = stats.ttest_ind(treat_pop, ctrl_pop, equal_var=False)
_, u_p = stats.mannwhitneyu(treat_pop, ctrl_pop, alternative="two-sided")
pooled_sd = np.sqrt(((treat_pop.std(ddof=1) ** 2) + (ctrl_pop.std(ddof=1) ** 2)) / 2)
cohens_d = (treat_pop.mean() - ctrl_pop.mean()) / pooled_sd if pooled_sd > 0 else 0.0


def bootstrap_ci(a, b, n_iter=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = [
        rng.choice(a, len(a), replace=True).mean()
        - rng.choice(b, len(b), replace=True).mean()
        for _ in range(n_iter)
    ]
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))


ci_low, ci_high = bootstrap_ci(treat_pop, ctrl_pop)
pop_result = {
    "control_mean": float(ctrl_pop.mean()),
    "treatment_mean": float(treat_pop.mean()),
    "diff": float(treat_pop.mean() - ctrl_pop.mean()),
    "ci": (ci_low, ci_high),
    "welch_p": float(t_p),
    "mannwhitney_p": float(u_p),
    "cohens_d": float(cohens_d),
}
for k, v in pop_result.items():
    print(f"  {k} = {v}")

fig = px.box(
    pd.concat([ctrl_pl.assign(arm="control"), treat_pl.assign(arm="treatment")]),
    x="arm",
    y="mean_popularity",
    points="all",
    title="Playlist popularity by arm",
)
fig.update_layout(height=480)
fig.show()

  control_mean = 18.7328
  treatment_mean = 17.11144444444444
  diff = -1.62135555555556
  ci = (-3.4188052777777798, 0.20533472222221935)
  welch_p = 0.08260186665363993
  mannwhitney_p = 0.6134932790515543
  cohens_d = -0.1769008485915937


In [9]:
ALPHA = -0.4
BETA_MATCH = 0.9
BETA_AUDIO = 0.6
BETA_POP = 0.25
NOISE_SD = 0.10
N_USERS = 2000
EXPOSURES_PER_USER = 4
TRACKS_PER_PLAYLIST = 25

scored_pdf = scored.to_pandas()
audio_z = scored_pdf[FEATURES].to_numpy()
audio_z = (audio_z - audio_z.mean(0)) / audio_z.std(0).clip(min=1e-9)
pop_z = (
    scored_pdf["popularity"].to_numpy() - scored_pdf["popularity"].mean()
) / scored_pdf["popularity"].std()
scored_pdf["audio_z"] = list(audio_z)
scored_pdf["pop_z"] = pop_z

In [10]:
def cosine(a, b):
    return float(
        np.dot(a, b) / ((np.linalg.norm(a) + 1e-9) * (np.linalg.norm(b) + 1e-9))
    )


def simulate_arm(pool_pdf, label_col, n_users, exposures, n_tracks, seed):
    rng = np.random.default_rng(seed)
    counts = pool_pdf[label_col].value_counts()
    eligible = counts[counts >= n_tracks * 2].index.tolist()
    rows = []
    for u in range(n_users):
        for _ in range(exposures):
            g = rng.choice(eligible)
            in_genre = pool_pdf[pool_pdf[label_col] == g]
            playlist = in_genre.sample(
                n=n_tracks, random_state=int(rng.integers(0, 2**31 - 1))
            )
            centroid = np.stack(playlist["audio_z"].to_numpy()).mean(0)
            track = playlist.sample(
                1, random_state=int(rng.integers(0, 2**31 - 1))
            ).iloc[0]
            align = cosine(track["audio_z"], centroid)
            logit = (
                ALPHA
                + BETA_MATCH * float(track[label_col] == g)
                + BETA_AUDIO * align
                + BETA_POP * track["pop_z"]
                + rng.normal(0, NOISE_SD)
            )
            rows.append({
                "user": u,
                "genre": g,
                "completed": int(rng.random() < 1 / (1 + np.exp(-logit))),
            })
    return pd.DataFrame(rows)


ctrl_sim = simulate_arm(
    scored_pdf, "genre", N_USERS, EXPOSURES_PER_USER, TRACKS_PER_PLAYLIST, SEED + 100
)
treat_sim = simulate_arm(
    scored_pdf[scored_pdf["confidence"] >= CONFIDENCE_TAU],
    "predicted_genre",
    N_USERS,
    EXPOSURES_PER_USER,
    TRACKS_PER_PLAYLIST,
    SEED + 200,
)
print(f"control_completion = {ctrl_sim['completed'].mean()}")
print(f"treatment_completion = {treat_sim['completed'].mean()}")

control_completion = 0.68525
treatment_completion = 0.70875


In [11]:
c_n, t_n = len(ctrl_sim), len(treat_sim)
c_c, t_c = int(ctrl_sim["completed"].sum()), int(treat_sim["completed"].sum())
z_stat, z_p = proportions_ztest(count=[t_c, c_c], nobs=[t_n, c_n])
ci_low_p, ci_high_p = confint_proportions_2indep(t_c, t_n, c_c, c_n, method="wald")


def bootstrap_rate_ci(a, b, n_iter=3000, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = [
        rng.choice(a, len(a), replace=True).mean()
        - rng.choice(b, len(b), replace=True).mean()
        for _ in range(n_iter)
    ]
    return float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))


boot_low, boot_high = bootstrap_rate_ci(
    treat_sim["completed"].to_numpy(), ctrl_sim["completed"].to_numpy()
)
eng_result = {
    "control_rate": c_c / c_n,
    "treatment_rate": t_c / t_n,
    "diff": t_c / t_n - c_c / c_n,
    "z": float(z_stat),
    "p": float(z_p),
    "wald_ci": (float(ci_low_p), float(ci_high_p)),
    "boot_ci": (boot_low, boot_high),
}
for k, v in eng_result.items():
    print(f"  {k} = {v}")

fig = px.bar(
    pd.DataFrame({
        "arm": ["control", "treatment"],
        "completion_rate": [eng_result["control_rate"], eng_result["treatment_rate"]],
    }),
    x="arm",
    y="completion_rate",
    text="completion_rate",
    title="Simulated completion rate by arm",
)
fig.update_traces(texttemplate="%{text}")
fig.update_layout(
    yaxis_range=[
        0,
        max(eng_result["control_rate"], eng_result["treatment_rate"]) * 1.2,
    ],
    height=440,
)
fig.show()

  control_rate = 0.68525
  treatment_rate = 0.70875
  diff = 0.023499999999999965
  z = 3.2341499702382044
  p = 0.0012200537157238934
  wald_ci = (0.009263153716563619, 0.037736846283436314)
  boot_ci = (0.009746874999999927, 0.037628125000000026)


In [12]:
summary = pl.from_pandas(
    pd.DataFrame([
        {
            "kpi": "popularity proxy",
            "control": pop_result["control_mean"],
            "treatment": pop_result["treatment_mean"],
            "diff": pop_result["diff"],
            "p": pop_result["welch_p"],
        },
        {
            "kpi": "completion rate",
            "control": eng_result["control_rate"],
            "treatment": eng_result["treatment_rate"],
            "diff": eng_result["diff"],
            "p": eng_result["p"],
        },
    ])
)
print(summary)
summary.write_parquet(DATA_DIR / "ab_summary.parquet")

shape: (2, 5)
┌──────────────────┬─────────┬───────────┬───────────┬──────────┐
│ kpi              ┆ control ┆ treatment ┆ diff      ┆ p        │
│ ---              ┆ ---     ┆ ---       ┆ ---       ┆ ---      │
│ str              ┆ f64     ┆ f64       ┆ f64       ┆ f64      │
╞══════════════════╪═════════╪═══════════╪═══════════╪══════════╡
│ popularity proxy ┆ 18.7328 ┆ 17.111444 ┆ -1.621356 ┆ 0.082602 │
│ completion rate  ┆ 0.68525 ┆ 0.70875   ┆ 0.0235    ┆ 0.00122  │
└──────────────────┴─────────┴───────────┴───────────┴──────────┘
